In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
from collections import defaultdict
import numpy as np


def get_pct_in_low(df):
    tau_mode = df["tau_hb_high"]
    lst = (tau_mode == True)
    first_index = None
    if lst.any():
        first_index = lst.idxmax()

    true_count = tau_mode.sum()
    false_count = (~tau_mode).sum()
    false_time = false_count * 10
    total_time = false_time + true_count * 30
    return (false_time / total_time, first_index)


def analyze_runs(runs):
    """
    Takes a list of run dicts and returns a list of result dicts, one per run:
      sf, bw, kp, ki, nodes,
      mean_hw_delay, std_hw_delay,
      pct_in_low  (fraction 0–1),
      settling_index  (first row where tau_hb_high is True, or None)
    """
    results = []
    for run in runs:
        stats    = pd.read_csv(run['main_stats'])
        hw_stats = pd.read_csv(run['hw_stats'])

        mean = hw_stats.mean()
        std  = hw_stats.std()
        pct_in_low, first_index = get_pct_in_low(stats)

        results.append({
            'sf':             run.get('sf'),
            'bw':             run.get('bw'),
            'kp':             run.get('kp'),
            'ki':             run.get('ki'),
            'nodes':          len(run.get('nodes', [])),
            'mean_hw_delay':  float(mean.iloc[0]),
            'std_hw_delay':   float(std.iloc[0]),
            'pct_in_low':     float(pct_in_low),
            'settling_index': first_index,
        })
    return results


def group_results(results, by=('sf', 'bw')):
    """
    Group a flat results list into an ordered dict keyed by tuples of `by` values.
    e.g. group_results(results, by=('sf', 'bw'))  →  {(7, 125): [...], (8, 125): [...]}
    """
    groups = defaultdict(list)
    for r in results:
        key = tuple(r[k] for k in by)
        groups[key].append(r)
    return dict(groups)


def aggregate_group(group):
    """
    Collapse a group of repeated runs (same SF/BW/…) into a single summary dict.
    Numeric metrics become mean ± std across the runs; metadata is taken from
    the first run (assumed identical across repetitions).
    """
    first = group[0]
    means = [r['mean_hw_delay']  for r in group]
    stds  = [r['std_hw_delay']   for r in group]
    plows = [r['pct_in_low']     for r in group]
    settles = [r['settling_index'] for r in group if r['settling_index'] is not None]
    return {
        'sf':               first['sf'],
        'bw':               first['bw'],
        'kp':               first['kp'],
        'ki':               first['ki'],
        'nodes':            first['nodes'],
        'n_runs':           len(group),
        'mean_hw_delay':    float(np.mean(means)),
        'mean_hw_delay_sd': float(np.std(means, ddof=1)) if len(means) > 1 else 0.0,
        'std_hw_delay':     float(np.mean(stds)),
        'pct_in_low':       float(np.mean(plows)),
        'settling_index':   int(np.mean(settles)) if settles else None,
    }


# ── Printing helpers ──────────────────────────────────────────────────────────

_COL_W = dict(run=4, sf=4, bw=5, kp=4, ki=4, nodes=5,
              mean=16, std=12, pct=10, settle=9)

def _table_header():
    cw = _COL_W
    return (
        f"{'#':>{cw['run']}}  "
        f"{'SF':>{cw['sf']}}  "
        f"{'BW':>{cw['bw']}}  "
        f"{'KP':>{cw['kp']}}  "
        f"{'KI':>{cw['ki']}}  "
        f"{'Nodes':>{cw['nodes']}}  "
        f"{'Mean(µs)':>{cw['mean']}}  "
        f"{'Std(µs)':>{cw['std']}}  "
        f"{'%InLow':>{cw['pct']}}  "
        f"{'Settle':>{cw['settle']}}"
    )

def _table_row(i, r):
    cw = _COL_W
    settle = str(r['settling_index']) if r['settling_index'] is not None else "—"
    # mean may carry a ±sd if this is an aggregated row
    sd_tag = f" ±{r['mean_hw_delay_sd']:.2f}" if 'mean_hw_delay_sd' in r else ""
    mean_str = f"{r['mean_hw_delay']:.3f}{sd_tag}"
    return (
        f"{i:>{cw['run']}}  "
        f"{str(r['sf']):>{cw['sf']}}  "
        f"{str(r['bw']):>{cw['bw']}}  "
        f"{str(r['kp']):>{cw['kp']}}  "
        f"{str(r['ki']):>{cw['ki']}}  "
        f"{r['nodes']:>{cw['nodes']}}  "
        f"{mean_str:>{cw['mean']}}  "
        f"{r['std_hw_delay']:>{cw['std']}.3f}  "
        f"{r['pct_in_low']*100:>{cw['pct']}.1f}%  "
        f"{settle:>{cw['settle']}}"
    )

def print_run_results(results, title=None):
    """Pretty-print a flat list of result dicts as a table."""
    header = _table_header()
    sep = "─" * len(header)
    if title:
        print(f"\n{'── ' + title + ' ':─<{len(header)}}")
    else:
        print(sep)
    print(header)
    print(sep)
    for i, r in enumerate(results, start=1):
        print(_table_row(i, r))

def print_grouped_results(results, by=('sf', 'bw'), aggregate=False):
    """
    Group results by `by`, then print each configuration as a titled block.

    aggregate=True  — collapse repeated runs within a group into one summary row
                      (mean ± std across runs).  Useful when you have multiple
                      repetitions of the same (SF, BW) config.
    aggregate=False — show every individual run inside each group (default).
    """
    grouped = group_results(results, by=by)
    for key, group in grouped.items():
        label = "  ".join(f"{k.upper()}={v}" for k, v in zip(by, key))
        if aggregate:
            print_run_results([aggregate_group(group)], title=f"{label}  (n={len(group)})")
        else:
            print_run_results(group, title=label)
        print()


In [ ]:
exp1 = pd.read_json('analysis/data/experiments/run_20260524_173641.json')
runs = exp1["experiments"]
results = analyze_runs(runs)

# One block per (SF, BW) — each run shown individually
print_grouped_results(results, by=('sf', 'bw'))

# If you have repeated runs per config and want a single summary row:
# print_grouped_results(results, by=('sf', 'bw'), aggregate=True)


In [2]:
# to use when the json is not ready yet
from analysis.scripts.retrieve_stats import load_and_weight_datasets

df = load_and_weight_datasets(file_pattern="analysis/data/main/main_stats_26-*.csv")
print_grouped_results()


Skipping analysis/data/main/main_stats_26-05:06.47_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
Skipping analysis/data/main/main_stats_26-05:06.26_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
Error reading analysis/data/main/main_stats_26-05:06.05_SF7_BW125_KP10_KI40_1nodes.csv: No columns to parse from file
Skipping analysis/data/main/main_stats_26-05:06.58_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
Skipping analysis/data/main/main_stats_26-05:07.08_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
Skipping analysis/data/main/main_stats_26-05:06.16_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
Skipping analysis/data/main/main_stats_26-05:06.37_SF7_BW125_KP10_KI40_1nodes.csv: Only 1 entries (needs > 10).
